In [ ]:
# --- Librerias ---
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"   # evita el choque de OpenMP (torch + cv2)
import time
import cv2
import numpy as np
import matplotlib.pyplot as plt
import torch
from segment_anything import sam_model_registry, SamAutomaticMaskGenerator

In [ ]:
# --- Parametros y rutas ---
tipo_modelo = "vit_h"
ruta_checkpoint = r"vit_h.pth"
ruta_imagen = r"datos/escritorio.jpg"
carpeta_salida = r"resultados"
dispositivo = "cuda" if torch.cuda.is_available() else "cpu"
os.makedirs(carpeta_salida, exist_ok=True)

In [ ]:
# --- Cargar la imagen ---
imagen_rgb = cv2.cvtColor(cv2.imread(ruta_imagen), cv2.COLOR_BGR2RGB)
alto, ancho = imagen_rgb.shape[:2]
pixeles_totales = alto * ancho

In [ ]:
# --- Cargar SAM y generar las mascaras ---
modelo_sam = sam_model_registry[tipo_modelo](checkpoint=ruta_checkpoint).to(dispositivo)
generador = SamAutomaticMaskGenerator(modelo_sam)
inicio = time.perf_counter()
mascaras = generador.generate(imagen_rgb)
print(f"{len(mascaras)} mascaras en {time.perf_counter() - inicio:.1f} s ({dispositivo})")

In [ ]:
# --- Quedarse con la mascara de mayor area ---
mascara_grande = max(mascaras, key=lambda m: m["area"])["segmentation"]
area_grande = int(mascara_grande.sum())
print(f"Mascara mayor: {area_grande} px ({100.0 * area_grande / mascara_grande.size:.1f} % de la imagen)")

In [ ]:
# --- Resaltar la mascara y guardar ---
resaltado = imagen_rgb.copy()
resaltado[mascara_grande] = (0.4 * resaltado[mascara_grande] + 0.6 * np.array([0, 255, 0], np.float32)).astype(np.uint8)
contornos, _ = cv2.findContours(mascara_grande.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
cv2.drawContours(resaltado, contornos, -1, (255, 0, 0), 2)
fig, ax = plt.subplots(1, 2, figsize=(14, 6))
ax[0].imshow(imagen_rgb); ax[0].set_title("Original"); ax[0].axis("off")
ax[1].imshow(resaltado);  ax[1].set_title(f"Mascara mas grande ({area_grande} px)"); ax[1].axis("off")
fig.tight_layout()
ruta_figura = os.path.join(carpeta_salida, "MascGrande_vit_h.png")
fig.savefig(ruta_figura, dpi=150, bbox_inches="tight")
plt.show()
print("Guardado:", ruta_figura)